In [11]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_executive_dashboard
#
# Layer
# -----
# Gold Layer - Data Products
#
# Purpose
# -------
# Build executive-ready KPI and alert datasets for senior
# leadership, Power BI dashboards and AI decision intelligence.
#
# Outputs
# -------
# • gold_executive_dashboard
# • gold_executive_alerts
#
# Executive Consumers
# -------------------
# • Chief Executive Officer
# • Chief Financial Officer
# • Chief Risk Officer
# • Chief Operating Officer
# • Board and Risk Committees
#
# Enterprise Concepts
# -------------------
# ✓ Executive KPI Layer
# ✓ Decision Intelligence
# ✓ Portfolio Risk Monitoring
# ✓ Scenario Analysis
# ✓ Automated Risk Alerts
# ✓ Power BI Data Product
# ✓ AI-ready Business Context
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

customer_portfolio_table = "gold_customer_portfolio"
regulatory_summary_table = "gold_regulatory_summary"

executive_dashboard_table = "gold_executive_dashboard"
executive_alerts_table = "gold_executive_alerts"

pipeline_name = "nb_build_executive_dashboard"

run_start_time = datetime.now()

print("ERIP Executive Dashboard Data Product Build Started")

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 14, Finished, Available, Finished, False)

ERIP Executive Dashboard Data Product Build Started


In [12]:
# ============================================================
# SECTION 2 - READ GOLD DATA PRODUCTS
# ============================================================
#
# Purpose
# -------
# Load the customer portfolio and regulatory summary datasets
# used to build executive KPIs and risk alerts.
#
# Input Grain
# -----------
# gold_customer_portfolio:
#     One row per customer
#
# gold_regulatory_summary:
#     One row per scenario month × scenario × IFRS stage
#     × country × industry
# ============================================================

gold_customer_portfolio = spark.table(customer_portfolio_table)

gold_regulatory_summary = spark.table(regulatory_summary_table)

print(f"Customer Portfolio Rows : {gold_customer_portfolio.count()}")
print(f"Regulatory Summary Rows : {gold_regulatory_summary.count()}")

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 15, Finished, Available, Finished, False)

Customer Portfolio Rows : 1000
Regulatory Summary Rows : 13824


In [13]:
# ============================================================
# SECTION 3 - CALCULATE PORTFOLIO-WIDE CUSTOMER KPIs
# ============================================================
#
# Purpose
# -------
# Calculate stable portfolio KPIs from the customer-level Gold
# data product.
#
# Measures
# --------
# • Total customers
# • Total loans
# • Total approved limit
# • Outstanding balance
# • Exposure
# • RWA
# • Base Expected Credit Loss
# • Average PD and LGD
# • High-risk customers
# • Strategic customers
# ============================================================

portfolio_kpis = (
    gold_customer_portfolio
    .agg(
        countDistinct("customer_id").alias("total_customers"),
        sum("number_of_loans").alias("total_loans"),
        sum("total_approved_limit").alias("total_approved_limit"),
        sum("total_outstanding_balance").alias("total_outstanding_balance"),
        sum("total_exposure").alias("total_exposure"),
        sum("total_rwa").alias("total_rwa"),
        sum("total_expected_credit_loss").alias("total_base_ecl"),
        avg("average_pd").alias("portfolio_average_pd"),
        avg("average_lgd").alias("portfolio_average_lgd"),
        avg("average_utilization").alias("portfolio_average_utilization"),
        sum(
            when(col("high_risk_flag") == "Y", 1).otherwise(0)
        ).alias("high_risk_customer_count"),
        sum(
            when(col("strategic_customer_flag") == "Y", 1).otherwise(0)
        ).alias("strategic_customer_count"),
        sum(
            when(col("highest_ifrs_stage") == 3, 1).otherwise(0)
        ).alias("stage_3_customer_count")
    )
)

display(portfolio_kpis)

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f28cf4c4-34b8-4d5b-8c47-f555b046703f)

In [14]:
# ============================================================
# SECTION 4 - IDENTIFY LEADING RISK CONCENTRATIONS
# ============================================================
#
# Purpose
# -------
# Identify the customers, countries and industries with the
# highest exposure and Expected Credit Loss.
#
# These attributes provide executive context and support:
#
# • Root-cause analysis
# • Power BI KPI commentary
# • AI-generated executive summaries
# • Portfolio concentration monitoring
# ============================================================

highest_exposure_customer = (
    gold_customer_portfolio
    .orderBy(col("total_exposure").desc())
    .select(
        col("customer_name").alias("highest_exposure_customer"),
        col("total_exposure").alias("highest_customer_exposure")
    )
    .limit(1)
)

highest_ecl_customer = (
    gold_customer_portfolio
    .orderBy(col("total_expected_credit_loss").desc())
    .select(
        col("customer_name").alias("highest_ecl_customer"),
        col("total_expected_credit_loss").alias("highest_customer_ecl")
    )
    .limit(1)
)

country_concentration = (
    gold_customer_portfolio
    .groupBy("country")
    .agg(
        sum("total_exposure").alias("country_exposure"),
        sum("total_expected_credit_loss").alias("country_ecl")
    )
)

highest_risk_country = (
    country_concentration
    .orderBy(col("country_ecl").desc())
    .select(
        col("country").alias("highest_risk_country"),
        col("country_exposure").alias("highest_country_exposure"),
        col("country_ecl").alias("highest_country_ecl")
    )
    .limit(1)
)

industry_concentration = (
    gold_customer_portfolio
    .groupBy("industry_name")
    .agg(
        sum("total_exposure").alias("industry_exposure"),
        sum("total_expected_credit_loss").alias("industry_ecl")
    )
)

highest_risk_industry = (
    industry_concentration
    .orderBy(col("industry_ecl").desc())
    .select(
        col("industry_name").alias("highest_risk_industry"),
        col("industry_exposure").alias("highest_industry_exposure"),
        col("industry_ecl").alias("highest_industry_ecl")
    )
    .limit(1)
)

print("✓ Executive concentration indicators calculated")

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 17, Finished, Available, Finished, False)

✓ Executive concentration indicators calculated


In [15]:
# ============================================================
# SECTION 5 - AGGREGATE MONTHLY SCENARIO KPIs
# ============================================================
#
# Purpose
# -------
# Aggregate the regulatory summary into one executive record
# for every scenario month and scenario.
#
# Grain
# -----
# One row per scenario month × scenario
#
# Measures
# --------
# • Base ECL
# • Stressed ECL
# • Incremental ECL
# • Capital impact
# • Average stressed PD and LGD
# • Minimum capital requirement
# ============================================================

scenario_kpis = (
    gold_regulatory_summary
    .groupBy(
        "scenario_sk",
        "scenario_id",
        "scenario_name",
        "scenario_rank",
        "scenario_severity",
        "scenario_month",
        "reporting_period",
        "stress_intensity",
        "reporting_currency",
        "regulatory_framework"
    )
    .agg(
        sum("stressed_ecl").alias("total_stressed_ecl"),
        sum("ecl_increase_amount").alias("total_ecl_increase"),
        sum("capital_impact_estimate").alias("total_capital_impact"),
        avg("average_ecl_increase_pct").alias("average_ecl_increase_pct"),
        avg("average_stressed_pd").alias("average_stressed_pd"),
        avg("average_stressed_lgd").alias("average_stressed_lgd")
    )
)

print(f"Executive scenario KPI rows: {scenario_kpis.count()}")
display(scenario_kpis.limit(10))

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 18, Finished, Available, Finished, False)

Executive scenario KPI rows: 108


SynapseWidget(Synapse.DataFrame, a212d5f6-2138-4ec7-ab3f-c3e825102dbd)

In [16]:
# ============================================================
# SECTION 6 - BUILD EXECUTIVE DASHBOARD DATA PRODUCT
# ============================================================
#
# Purpose
# -------
# Combine portfolio-wide KPIs, scenario measures and leading
# concentration indicators into one reporting-ready dataset.
#
# Grain
# -----
# One row per scenario month × scenario
#
# Derived Executive Indicators
# ----------------------------
# • Portfolio health status
# • Credit risk level
# • Capital pressure level
# • Scenario loss ratio
# • High-risk customer percentage
# ============================================================

gold_executive_dashboard = (
    scenario_kpis
    .crossJoin(portfolio_kpis)
    .crossJoin(highest_exposure_customer)
    .crossJoin(highest_ecl_customer)
    .crossJoin(highest_risk_country)
    .crossJoin(highest_risk_industry)
    .withColumn(
        "minimum_capital_requirement",
        col("total_rwa") * lit(0.08)
    )
    .withColumn(
        "ecl_to_exposure_pct",
        when(
            col("total_exposure") > 0,
            (col("total_base_ecl") / col("total_exposure")) * 100
        ).otherwise(0)
    )
    .withColumn(
        "stressed_ecl_to_exposure_pct",
        when(
            col("total_exposure") > 0,
            (col("total_stressed_ecl") / col("total_exposure")) * 100
        ).otherwise(0)
    )
    .withColumn(
        "high_risk_customer_pct",
        when(
            col("total_customers") > 0,
            (col("high_risk_customer_count") / col("total_customers")) * 100
        ).otherwise(0)
    )
    .withColumn(
        "portfolio_health_status",
        when(
            (col("high_risk_customer_pct") >= 30) |
            (col("stressed_ecl_to_exposure_pct") >= 10),
            "Critical"
        )
        .when(
            (col("high_risk_customer_pct") >= 15) |
            (col("stressed_ecl_to_exposure_pct") >= 5),
            "At Risk"
        )
        .when(
            (col("high_risk_customer_pct") >= 5) |
            (col("stressed_ecl_to_exposure_pct") >= 2),
            "Watch"
        )
        .otherwise("Stable")
    )
    .withColumn(
        "credit_risk_level",
        when(col("average_stressed_pd") >= 0.20, "Critical")
        .when(col("average_stressed_pd") >= 0.10, "High")
        .when(col("average_stressed_pd") >= 0.05, "Moderate")
        .otherwise("Low")
    )
    .withColumn(
        "capital_pressure_level",
        when(
            col("total_capital_impact") >=
            col("minimum_capital_requirement") * lit(0.25),
            "Severe"
        )
        .when(
            col("total_capital_impact") >=
            col("minimum_capital_requirement") * lit(0.10),
            "Elevated"
        )
        .otherwise("Normal")
    )
    .withColumn(
        "executive_alert_status",
        when(
            (col("portfolio_health_status") == "Critical") |
            (col("credit_risk_level") == "Critical") |
            (col("capital_pressure_level") == "Severe"),
            "Red"
        )
        .when(
            (col("portfolio_health_status").isin("At Risk", "Watch")) |
            (col("credit_risk_level").isin("High", "Moderate")) |
            (col("capital_pressure_level") == "Elevated"),
            "Amber"
        )
        .otherwise("Green")
    )
    .select(
        "scenario_sk",
        "scenario_id",
        "scenario_name",
        "scenario_rank",
        "scenario_severity",
        "scenario_month",
        "reporting_period",
        "stress_intensity",
        "reporting_currency",
        "regulatory_framework",

        "total_customers",
        "total_loans",
        "total_approved_limit",
        "total_outstanding_balance",
        "total_exposure",
        "total_rwa",
        "total_base_ecl",
        "portfolio_average_pd",
        "portfolio_average_lgd",
        "portfolio_average_utilization",
        "high_risk_customer_count",
        "high_risk_customer_pct",
        "strategic_customer_count",
        "stage_3_customer_count",

        "total_stressed_ecl",
        "total_ecl_increase",
        "average_ecl_increase_pct",
        "total_capital_impact",
        "average_stressed_pd",
        "average_stressed_lgd",
        "minimum_capital_requirement",

        "ecl_to_exposure_pct",
        "stressed_ecl_to_exposure_pct",
        "portfolio_health_status",
        "credit_risk_level",
        "capital_pressure_level",
        "executive_alert_status",

        "highest_exposure_customer",
        "highest_customer_exposure",
        "highest_ecl_customer",
        "highest_customer_ecl",
        "highest_risk_country",
        "highest_country_exposure",
        "highest_country_ecl",
        "highest_risk_industry",
        "highest_industry_exposure",
        "highest_industry_ecl",

        current_timestamp().alias("gold_updated_timestamp")
    )
)

print(f"Executive dashboard rows created: {gold_executive_dashboard.count()}")
display(gold_executive_dashboard.limit(10))

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 19, Finished, Available, Finished, False)

Executive dashboard rows created: 108


SynapseWidget(Synapse.DataFrame, 89173047-d5c5-4eea-bafe-8e6f1ca1ed19)

In [17]:
# ============================================================
# SECTION 7 - BUILD EXECUTIVE ALERTS DATA PRODUCT
# ============================================================
#
# Purpose
# -------
# Convert executive KPI conditions into structured alerts that
# can be consumed by Power BI, orchestration and AI agents.
#
# Grain
# -----
# One row per scenario month × scenario × alert type
#
# Alert Types
# -----------
# • Portfolio Health
# • Credit Risk
# • Capital Pressure
# • Customer Risk Concentration
# • Country Concentration
# • Industry Concentration
# ============================================================

portfolio_health_alerts = (
    gold_executive_dashboard
    .select(
        "scenario_sk",
        "scenario_name",
        "scenario_month",
        "reporting_period",
        lit("PORTFOLIO_HEALTH").alias("alert_type"),
        col("portfolio_health_status").alias("alert_severity"),
        concat(
            lit("Portfolio health is "),
            col("portfolio_health_status"),
            lit(" under the "),
            col("scenario_name"),
            lit(" scenario.")
        ).alias("alert_message"),
        col("stressed_ecl_to_exposure_pct").alias("alert_metric_value"),
        lit("Stressed ECL to Exposure %").alias("alert_metric_name")
    )
)

credit_risk_alerts = (
    gold_executive_dashboard
    .select(
        "scenario_sk",
        "scenario_name",
        "scenario_month",
        "reporting_period",
        lit("CREDIT_RISK").alias("alert_type"),
        col("credit_risk_level").alias("alert_severity"),
        concat(
            lit("Average stressed PD indicates "),
            col("credit_risk_level"),
            lit(" credit risk.")
        ).alias("alert_message"),
        (col("average_stressed_pd") * 100).alias("alert_metric_value"),
        lit("Average Stressed PD %").alias("alert_metric_name")
    )
)

capital_pressure_alerts = (
    gold_executive_dashboard
    .select(
        "scenario_sk",
        "scenario_name",
        "scenario_month",
        "reporting_period",
        lit("CAPITAL_PRESSURE").alias("alert_type"),
        col("capital_pressure_level").alias("alert_severity"),
        concat(
            lit("Capital pressure is "),
            col("capital_pressure_level"),
            lit(" under the "),
            col("scenario_name"),
            lit(" scenario.")
        ).alias("alert_message"),
        col("total_capital_impact").alias("alert_metric_value"),
        lit("Estimated Capital Impact").alias("alert_metric_name")
    )
)

customer_risk_alerts = (
    gold_executive_dashboard
    .select(
        "scenario_sk",
        "scenario_name",
        "scenario_month",
        "reporting_period",
        lit("HIGH_RISK_CUSTOMERS").alias("alert_type"),
        when(col("high_risk_customer_pct") >= 30, "Critical")
        .when(col("high_risk_customer_pct") >= 15, "High")
        .when(col("high_risk_customer_pct") >= 5, "Moderate")
        .otherwise("Low")
        .alias("alert_severity"),
        concat(
            format_number(col("high_risk_customer_pct"), 2),
            lit("% of customers are classified as high risk.")
        ).alias("alert_message"),
        col("high_risk_customer_pct").alias("alert_metric_value"),
        lit("High Risk Customer %").alias("alert_metric_name")
    )
)

country_concentration_alerts = (
    gold_executive_dashboard
    .select(
        "scenario_sk",
        "scenario_name",
        "scenario_month",
        "reporting_period",
        lit("COUNTRY_CONCENTRATION").alias("alert_type"),
        when(
            col("highest_country_exposure") / col("total_exposure") >= 0.25,
            "High"
        ).when(
            col("highest_country_exposure") / col("total_exposure") >= 0.15,
            "Moderate"
        ).otherwise("Low")
        .alias("alert_severity"),
        concat(
            col("highest_risk_country"),
            lit(" is the largest country risk concentration.")
        ).alias("alert_message"),
        (
            col("highest_country_exposure") /
            col("total_exposure") * 100
        ).alias("alert_metric_value"),
        lit("Highest Country Exposure %").alias("alert_metric_name")
    )
)

industry_concentration_alerts = (
    gold_executive_dashboard
    .select(
        "scenario_sk",
        "scenario_name",
        "scenario_month",
        "reporting_period",
        lit("INDUSTRY_CONCENTRATION").alias("alert_type"),
        when(
            col("highest_industry_exposure") / col("total_exposure") >= 0.20,
            "High"
        ).when(
            col("highest_industry_exposure") / col("total_exposure") >= 0.10,
            "Moderate"
        ).otherwise("Low")
        .alias("alert_severity"),
        concat(
            col("highest_risk_industry"),
            lit(" is the largest industry risk concentration.")
        ).alias("alert_message"),
        (
            col("highest_industry_exposure") /
            col("total_exposure") * 100
        ).alias("alert_metric_value"),
        lit("Highest Industry Exposure %").alias("alert_metric_name")
    )
)

gold_executive_alerts = (
    portfolio_health_alerts
    .unionByName(credit_risk_alerts)
    .unionByName(capital_pressure_alerts)
    .unionByName(customer_risk_alerts)
    .unionByName(country_concentration_alerts)
    .unionByName(industry_concentration_alerts)
    .withColumn(
        "requires_action",
        when(
            col("alert_severity").isin(
                "Critical",
                "High",
                "Severe",
                "At Risk"
            ),
            "Y"
        ).otherwise("N")
    )
    .withColumn(
        "alert_generated_timestamp",
        current_timestamp()
    )
)

print(f"Executive alert rows created: {gold_executive_alerts.count()}")
display(gold_executive_alerts.limit(20))

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 20, Finished, Available, Finished, False)

Executive alert rows created: 648


SynapseWidget(Synapse.DataFrame, 36800f97-9642-4c0b-9c33-854e58397074)

In [18]:
# ============================================================
# SECTION 8 - BUILD MANAGEMENT ATTENTION DATA PRODUCT
# ============================================================
#
# Purpose
# -------
# Create a current executive-facing alert layer from the
# historical gold_executive_alerts data product.
#
# Grain
# -----
# One row per alert type for the latest reporting period.
#
# Logic
# -----
# 1. Keep only the latest reporting period
# 2. Prefer the most severe alert
# 3. Where severity is equal, prefer the more severe scenario
# 4. Keep one alert per alert type
# 5. Retain only alerts requiring management action
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ------------------------------------------------------------
# 1. Severity ranking
# ------------------------------------------------------------

alerts_ranked = (
    gold_executive_alerts
    .withColumn(
        "severity_rank",
        F.when(F.col("alert_severity") == "Critical", 5)
         .when(F.col("alert_severity") == "Severe", 5)
         .when(F.col("alert_severity") == "At Risk", 5)
         .when(F.col("alert_severity") == "High", 4)
         .when(F.col("alert_severity") == "Moderate", 3)
         .when(F.col("alert_severity") == "Elevated", 3)
         .when(F.col("alert_severity") == "Low", 1)
         .otherwise(0)
    )
)

# ------------------------------------------------------------
# 2. Scenario ranking
# ------------------------------------------------------------

alerts_ranked = (
    alerts_ranked
    .withColumn(
        "scenario_severity_rank",
        F.when(F.col("scenario_name") == "Severe", 3)
         .when(F.col("scenario_name") == "Adverse", 2)
         .when(F.col("scenario_name") == "Baseline", 1)
         .otherwise(0)
    )
)

# ------------------------------------------------------------
# 3. Identify latest reporting period
# ------------------------------------------------------------

latest_reporting_period = (
    alerts_ranked
    .agg(F.max("reporting_period").alias("latest_period"))
    .first()["latest_period"]
)

current_alerts = (
    alerts_ranked
    .filter(F.col("reporting_period") == latest_reporting_period)
)

# ------------------------------------------------------------
# 4. Pick strongest/current alert per alert type
# ------------------------------------------------------------

alert_window = (
    Window
    .partitionBy("alert_type")
    .orderBy(
        F.col("severity_rank").desc(),
        F.col("scenario_severity_rank").desc(),
        F.col("alert_metric_value").desc()
    )
)

gold_management_attention = (
    current_alerts
    .withColumn(
        "alert_priority",
        F.row_number().over(alert_window)
    )
    .filter(F.col("alert_priority") == 1)
    .filter(F.col("requires_action") == "Y")
)

# ------------------------------------------------------------
# 5. Add executive-friendly display title
# ------------------------------------------------------------

gold_management_attention = (
    gold_management_attention
    .withColumn(
        "alert_title",
        F.when(
            F.col("alert_type") == "PORTFOLIO_HEALTH",
            F.lit("Portfolio Health")
        )
        .when(
            F.col("alert_type") == "CREDIT_RISK",
            F.lit("Credit Risk")
        )
        .when(
            F.col("alert_type") == "CAPITAL_PRESSURE",
            F.lit("Capital Pressure")
        )
        .when(
            F.col("alert_type") == "HIGH_RISK_CUSTOMERS",
            F.lit("High-Risk Customer Concentration")
        )
        .when(
            F.col("alert_type") == "COUNTRY_CONCENTRATION",
            F.lit("Country Concentration")
        )
        .when(
            F.col("alert_type") == "INDUSTRY_CONCENTRATION",
            F.lit("Industry Concentration")
        )
        .otherwise(F.col("alert_type"))
    )
)

# ------------------------------------------------------------
# 6. Final executive-facing schema
# ------------------------------------------------------------

gold_management_attention = (
    gold_management_attention
    .select(
        "alert_title",
        "alert_type",
        "alert_severity",
        "alert_message",
        "alert_metric_name",
        "alert_metric_value",
        "scenario_name",
        "reporting_period",
        "requires_action",
        "alert_generated_timestamp"
    )
)

print(
    f"Management attention rows created: "
    f"{gold_management_attention.count()}"
)

display(gold_management_attention)

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 21, Finished, Available, Finished, False)

Management attention rows created: 4


SynapseWidget(Synapse.DataFrame, 438ff692-33fe-48cf-9414-1b9bcb06a796)

In [19]:
# ============================================================
# SECTION 8 - EXECUTIVE DATA PRODUCT QUALITY VALIDATION
# ============================================================
#
# Validation Checks
# -----------------
# • Scenario identifiers populated
# • Reporting periods populated
# • No negative portfolio or ECL values
# • One executive row per scenario month and scenario
# • Alerts created for every executive scenario record
# ============================================================

dashboard_rows = gold_executive_dashboard.count()

duplicate_dashboard_rows = (
    dashboard_rows -
    gold_executive_dashboard
    .select(
        "scenario_sk",
        "scenario_month"
    )
    .distinct()
    .count()
)

null_scenario_sk = gold_executive_dashboard.filter(
    col("scenario_sk").isNull()
).count()

null_reporting_period = gold_executive_dashboard.filter(
    col("reporting_period").isNull()
).count()

negative_exposure = gold_executive_dashboard.filter(
    col("total_exposure") < 0
).count()

negative_base_ecl = gold_executive_dashboard.filter(
    col("total_base_ecl") < 0
).count()

negative_stressed_ecl = gold_executive_dashboard.filter(
    col("total_stressed_ecl") < 0
).count()

alert_rows = gold_executive_alerts.count()

expected_alert_rows = dashboard_rows * 6

print("Executive Dashboard Quality Checks")
print("----------------------------------")
print(f"Dashboard Rows             : {dashboard_rows}")
print(f"Duplicate Dashboard Rows   : {duplicate_dashboard_rows}")
print(f"Null Scenario SK           : {null_scenario_sk}")
print(f"Null Reporting Period      : {null_reporting_period}")
print(f"Negative Exposure          : {negative_exposure}")
print(f"Negative Base ECL          : {negative_base_ecl}")
print(f"Negative Stressed ECL      : {negative_stressed_ecl}")
print(f"Executive Alert Rows       : {alert_rows}")
print(f"Expected Alert Rows        : {expected_alert_rows}")

if (
    duplicate_dashboard_rows > 0 or
    null_scenario_sk > 0 or
    null_reporting_period > 0 or
    negative_exposure > 0 or
    negative_base_ecl > 0 or
    negative_stressed_ecl > 0 or
    alert_rows != expected_alert_rows
):
    raise Exception("Executive Data Product Validation Failed")
else:
    print("✓ Executive Data Product Validation Passed")

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 22, Finished, Available, Finished, False)

Executive Dashboard Quality Checks
----------------------------------
Dashboard Rows             : 108
Duplicate Dashboard Rows   : 0
Null Scenario SK           : 0
Null Reporting Period      : 0
Negative Exposure          : 0
Negative Base ECL          : 0
Negative Stressed ECL      : 0
Executive Alert Rows       : 648
Expected Alert Rows        : 648
✓ Executive Data Product Validation Passed


In [20]:
# ============================================================
# SECTION 9 - WRITE GOLD EXECUTIVE DATA PRODUCTS
# ============================================================
#
# Outputs
# -------
# gold_executive_dashboard
# gold_executive_alerts
# ============================================================

(
    gold_executive_dashboard.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(executive_dashboard_table)
)

(
    gold_executive_alerts.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(executive_alerts_table)
)

# Persist newly created Management Attention data product
(
    gold_management_attention.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("gold_management_attention")
)

print("✓ Gold management attention created: gold_management_attention")
print(
    f"Management attention rows written: "
    f"{gold_management_attention.count()}"
)

print(
    f"✓ Gold executive dashboard created: "
    f"{executive_dashboard_table}"
)

print(
    f"Dashboard rows written: "
    f"{gold_executive_dashboard.count()}"
)

print(
    f"✓ Gold executive alerts created: "
    f"{executive_alerts_table}"
)

print(
    f"Alert rows written: "
    f"{gold_executive_alerts.count()}"
)

StatementMeta(, 2ce5927a-0f9e-45a1-bedc-222bc4132edc, 23, Finished, Available, Finished, False)

✓ Gold management attention created: gold_management_attention
Management attention rows written: 4
✓ Gold executive dashboard created: gold_executive_dashboard
Dashboard rows written: 108
✓ Gold executive alerts created: gold_executive_alerts
Alert rows written: 648
